# Process COM Antitrust files — v3

This notebook keeps the COM antitrust manifest, folder, identifier and
output conventions, while using the same document-extraction standard as
`process_com_manual_files_v3`.

Extraction design:

- HTML/TXT:
  - BOM and declared HTML/XML encoding
  - UnicodeDammit
  - charset-normalizer candidates
  - strict UTF-8 plus conservative CP1252/Latin-1 fallbacks
  - optional ftfy repair
  - scored best-content-container selection

- PDF:
  - PyMuPDF sorted text
  - PyMuPDF sorted blocks
  - optional `pdftotext -layout`
  - quality comparison across native candidates
  - MinerU only if the best native extraction is genuinely suspicious

- Outputs:
  - canonical regex/LLM-ready `.txt`
  - readable `.txt`
  - Markdown
  - clean manifest
  - candidate manifest
  - failed-row manifest

The canonical downstream text remains inside:

`data/processed/com_antitrust/`


## 1. Configuration and optional dependencies

In [1]:

from __future__ import annotations

import os
import re
import json
import shutil
import hashlib
import subprocess
import unicodedata
import tempfile
from collections import Counter
from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, Any, List, Tuple

import pandas as pd
from tqdm.auto import tqdm

try:
    import fitz  # PyMuPDF
except Exception:
    fitz = None

try:
    from bs4 import BeautifulSoup, UnicodeDammit
except Exception:
    BeautifulSoup = None
    UnicodeDammit = None

try:
    from charset_normalizer import from_bytes as charset_from_bytes
except Exception:
    charset_from_bytes = None

try:
    from ftfy import fix_text as ftfy_fix_text
except Exception:
    ftfy_fix_text = None

# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd().resolve()

PROJECT_ROOT = NOTEBOOK_DIR
for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (candidate / "data").exists() and (candidate / "output").exists():
        PROJECT_ROOT = candidate
        break

# Override manually if required:
# PROJECT_ROOT = Path("/home/edik/projects/eccjeu").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw" / "com_antitrust"
CLEAN_DATA_DIR = DATA_DIR / "processed" / "com_antitrust"
CANDIDATE_DATA_DIR = DATA_DIR / "processed" / "com_antitrust_candidates"
ERROR_DIR = CLEAN_DATA_DIR / "errors"
OUTPUT_DIR = PROJECT_ROOT / "output" / "com_antitrust"

DOWNLOAD_MANIFEST = OUTPUT_DIR / "download_manifest.csv"
FALLBACK_DOWNLOAD_MANIFESTS = [
    PROJECT_ROOT / "output" / "com_antitrust" / "download_manifest.csv",
    PROJECT_ROOT / "download_manifest.csv",
]

CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)
CANDIDATE_DATA_DIR.mkdir(parents=True, exist_ok=True)
ERROR_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_FILE_MANIFEST_BASENAME = "com_antitrust_clean_file_manifest"
CLEAN_FILE_MANIFEST_PATH = OUTPUT_DIR / f"{CLEAN_FILE_MANIFEST_BASENAME}.csv"
CLEAN_FILE_MANIFEST_XLSX_PATH = OUTPUT_DIR / f"{CLEAN_FILE_MANIFEST_BASENAME}.xlsx"
CANDIDATE_MANIFEST_PATH = OUTPUT_DIR / "com_antitrust_extraction_candidate_manifest.csv"
FAILED_FILE_MANIFEST_PATH = OUTPUT_DIR / "com_antitrust_failed_cleaning_rows.csv"

# ---------------------------------------------------------------------
# Production defaults
# ---------------------------------------------------------------------
EXTRACTION_VERSION = 3
FORCE_REEXTRACT = True
PROCESS_LIMIT = None

# Candidate diagnostics are always retained in the candidate manifest.
# Saving every candidate as separate raw/readable/regex text files is useful
# for debugging but adds significant disk I/O.
SAVE_ALL_CANDIDATES = False

RUN_PDFTOTEXT = True
RUN_MINERU = True
RUN_MINERU_FOR_ALL_PDFS = False
MINERU_TIMEOUT_SECONDS = 600

MIN_CHARS_ANY = 100
MIN_PDF_CHARS_GOOD = 1200
MIN_SCORE_ACCEPTABLE = 35.0
MIN_SCORE_OK = 55.0

# Conservative OCR policy: OCR is considered only after all native PDF
# candidates have been compared.
MIN_NATIVE_SCORE_TO_RUN_OCR = 45.0
SERIOUS_OCR_REASONS = {
    "very_short_text",
    "replacement_characters",
    "possible_mojibake",
    "private_use_characters",
    "repeated_character_garbage",
    "low_language_plausibility",
}

PDFTOTEXT_CLI = shutil.which("pdftotext")
MINERU_CLI = shutil.which("mineru") or shutil.which("magic-pdf")

if MINERU_CLI:
    MINERU_COMMAND_TEMPLATE = [
        MINERU_CLI,
        "-p", "{input_pdf}",
        "-o", "{output_dir}",
        "-m", "ocr",
        "-b", "pipeline",
    ]
else:
    MINERU_COMMAND_TEMPLATE = [
        "mineru",
        "-p", "{input_pdf}",
        "-o", "{output_dir}",
        "-m", "ocr",
        "-b", "pipeline",
    ]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DOWNLOAD_MANIFEST:", DOWNLOAD_MANIFEST)
print("Canonical clean directory:", CLEAN_DATA_DIR)
print("Candidate directory:", CANDIDATE_DATA_DIR)
print("PyMuPDF available:", fitz is not None)
print("BeautifulSoup available:", BeautifulSoup is not None)
print("charset-normalizer available:", charset_from_bytes is not None)
print("ftfy available:", ftfy_fix_text is not None)
print("pdftotext:", PDFTOTEXT_CLI)
print("MinerU:", MINERU_CLI)
print("FORCE_REEXTRACT:", FORCE_REEXTRACT)
print("SAVE_ALL_CANDIDATES:", SAVE_ALL_CANDIDATES)


PROJECT_ROOT: /home/edik/projects/eccjeu
DOWNLOAD_MANIFEST: /home/edik/projects/eccjeu/output/com_antitrust/download_manifest.csv
Canonical clean directory: /home/edik/projects/eccjeu/data/processed/com_antitrust
Candidate directory: /home/edik/projects/eccjeu/data/processed/com_antitrust_candidates
PyMuPDF available: True
BeautifulSoup available: True
charset-normalizer available: True
ftfy available: True
pdftotext: /usr/bin/pdftotext
MinerU: /home/edik/projects/.venv/bin/mineru
FORCE_REEXTRACT: True
SAVE_ALL_CANDIDATES: False


Optional installation cell. Run only if packages are missing. `pdftotext` is normally installed through the operating system rather than pip.

In [2]:

# Uncomment if needed:
# %pip install -U beautifulsoup4 charset-normalizer ftfy pymupdf openpyxl

# Ubuntu/Debian system package for pdftotext:
# !sudo apt-get update && sudo apt-get install -y poppler-utils


## 2. Load the download manifest and construct the work manifest

In [3]:

TEXT_EXTENSIONS = {".txt", ".text"}
HTML_EXTENSIONS = {".html", ".htm", ".xhtml"}
PDF_EXTENSIONS = {".pdf"}
SUPPORTED_EXTENSIONS = (
    TEXT_EXTENSIONS | HTML_EXTENSIONS | PDF_EXTENSIONS
)

PATH_COLUMNS = [
    "download_path", "final_local_path", "local_path",
    "target_path", "file_path", "path",
]
URL_COLUMNS = ["download_url", "url", "source_url"]
TITLE_COLUMNS = [
    "title", "item_label", "decision_title",
    "document_title", "name",
]
DOC_ID_COLUMNS = [
    "download_key", "identifier", "document_id",
    "source_entry_id", "id",
]
CASE_COLUMNS = [
    "case_number", "case_numbers", "case",
    "case_no", "case_id",
]


def resolve_manifest_path() -> Path:
    if DOWNLOAD_MANIFEST.exists():
        return DOWNLOAD_MANIFEST
    for path in FALLBACK_DOWNLOAD_MANIFESTS:
        if path.exists():
            return path
    raise FileNotFoundError(
        "Could not find the COM antitrust download manifest. "
        f"Expected at: {DOWNLOAD_MANIFEST}"
    )


def as_bool_series(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes", "y"})
    )


def first_existing_col(
    dataframe: pd.DataFrame,
    candidates: List[str],
) -> Optional[str]:
    for column in candidates:
        if column in dataframe.columns:
            return column
    return None


def first_nonempty(
    row: pd.Series,
    candidates: List[str],
) -> str:
    for column in candidates:
        if column in row.index:
            value = row.get(column)
            if (
                pd.notna(value)
                and str(value).strip()
                and str(value).strip().lower()
                not in {"nan", "none", "null"}
            ):
                return str(value).strip()
    return ""


def normalize_possible_path(value: str) -> Optional[Path]:
    if (
        not value
        or str(value).strip().lower()
        in {"nan", "none", "null"}
    ):
        return None

    raw = str(value).strip()
    path = Path(raw)

    candidates = []
    if path.is_absolute():
        candidates.append(path)
    else:
        candidates.extend([
            PROJECT_ROOT / path,
            DATA_DIR / path,
            RAW_DIR / path,
            OUTPUT_DIR / path,
        ])

    for candidate in candidates:
        if candidate.exists() and candidate.is_file():
            return candidate.resolve()
    return None


def infer_file_format(
    path: Optional[Path],
    row_format: str = "",
) -> str:
    if path is not None and path.suffix:
        extension = path.suffix.lower()
        if extension in PDF_EXTENSIONS:
            return "pdf"
        if extension in HTML_EXTENSIONS:
            return "html"
        if extension in TEXT_EXTENSIONS:
            return "txt"
        return extension.lstrip(".")
    return str(row_format or "unknown").lower()


def safe_slug(value: str, max_len: int = 80) -> str:
    value = str(value or "").strip()
    value = re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        value,
    ).strip("_")
    return value[:max_len] or "document"


def make_document_id(
    row: pd.Series,
    raw_path: Path,
) -> str:
    download_key = first_nonempty(row, ["download_key"])
    identifier = first_nonempty(row, ["identifier"])
    case_number = first_nonempty(row, CASE_COLUMNS)
    item_date = first_nonempty(
        row,
        ["item_date", "decision_date", "date"],
    )
    item_label = first_nonempty(
        row,
        ["item_label", "title", "decision_title"],
    )

    parts = [
        safe_slug(value)
        for value in [
            identifier,
            case_number,
            item_date,
            item_label,
            download_key,
        ]
        if value
    ]

    base = "__".join(parts[:5]) if parts else raw_path.stem
    if len(base) > 180:
        base = base[:150] + "__" + safe_slug(
            download_key or raw_path.stem,
            25,
        )

    digest = hashlib.sha1(
        str(raw_path).encode(
            "utf-8",
            errors="ignore",
        )
    ).hexdigest()[:10]

    file_format = infer_file_format(raw_path)
    return (
        f"com_antitrust__{safe_slug(base, 180)}"
        f"__{file_format}__{digest}"
    )


manifest_path = resolve_manifest_path()
df = pd.read_csv(
    manifest_path,
    low_memory=False,
)
df.columns = [
    str(column).strip()
    for column in df.columns
]

path_col = first_existing_col(df, PATH_COLUMNS)
url_col = first_existing_col(df, URL_COLUMNS)
success_col = first_existing_col(
    df,
    [
        "download_success",
        "success",
        "final_success",
    ],
)

if path_col is None:
    raise ValueError(
        "Manifest must contain a download/local-path column."
    )

download_success = (
    as_bool_series(df[success_col])
    if success_col
    else pd.Series(True, index=df.index)
)

resolved_paths = df[path_col].apply(
    normalize_possible_path
)
file_exists_now = resolved_paths.notna()

has_download_object = (
    df[url_col].notna()
    if url_col
    else pd.Series(False, index=df.index)
) | df[path_col].notna()

keep = (
    has_download_object
    & (download_success | file_exists_now)
)

work = df.loc[keep].copy()
work["__resolved_path"] = (
    resolved_paths.loc[work.index]
)
work["__download_success"] = (
    download_success.loc[work.index].values
)

records = []
for _, row in work.iterrows():
    raw_path = row.get("__resolved_path")
    file_format = infer_file_format(
        raw_path,
        row.get("file_format", ""),
    )

    document_id = (
        make_document_id(row, raw_path)
        if raw_path is not None
        else ""
    )

    records.append({
        "document_id": document_id,
        "download_key": first_nonempty(
            row,
            ["download_key"],
        ),
        "identifier": first_nonempty(
            row,
            ["identifier"],
        ),
        "case_number": first_nonempty(
            row,
            CASE_COLUMNS,
        ),
        "item_date": first_nonempty(
            row,
            ["item_date", "decision_date", "date"],
        ),
        "item_label": first_nonempty(
            row,
            [
                "item_label",
                "title",
                "decision_title",
            ],
        ),
        "title": first_nonempty(
            row,
            TITLE_COLUMNS,
        ),
        "file_format": file_format,
        "download_url": (
            first_nonempty(row, URL_COLUMNS)
            if url_col
            else ""
        ),
        "download_success": bool(
            row.get("__download_success", False)
        ),
        "download_status": row.get(
            "download_status",
            "",
        ),
        "raw_file_exists": raw_path is not None,
        "raw_file_path": (
            str(raw_path)
            if raw_path is not None
            else str(row.get(path_col, ""))
        ),
    })

work_manifest = pd.DataFrame(records)
work_manifest = (
    work_manifest
    .drop_duplicates(
        subset=["raw_file_path", "file_format"],
        keep="first",
    )
    .reset_index(drop=True)
)

work_manifest["supported_file_type"] = (
    work_manifest["raw_file_path"].apply(
        lambda value: (
            Path(str(value)).suffix.lower()
            in SUPPORTED_EXTENSIONS
            if str(value).strip()
            else False
        )
    )
)
work_manifest["cleaning_target"] = (
    work_manifest["raw_file_exists"]
    & work_manifest["supported_file_type"]
)

print("Loaded:", manifest_path)
print("Rows in source manifest:", len(df))
print("Rows in work manifest:", len(work_manifest))
print(
    "Cleaning targets:",
    int(work_manifest["cleaning_target"].sum()),
)
display(
    work_manifest[
        [
            "download_key",
            "case_number",
            "raw_file_path",
            "file_format",
            "cleaning_target",
        ]
    ].head(20)
)


Loaded: /home/edik/projects/eccjeu/output/com_antitrust/download_manifest.csv
Rows in source manifest: 2492
Rows in work manifest: 1174
Cleaning targets: 1174


,download_key,case_number,raw_file_path,file_format,cleaning_target
0,055d7039aa0f89ee3daa,AT.28841,/home/edik/projects/eccjeu/data/raw/com_antitr...,pdf,True
1,432056ba775eb3636794,AT.29629,/home/edik/projects/eccjeu/data/raw/com_antitr...,pdf,True
2,4e58062cea9b40420c02,AT.30373,/home/edik/projects/eccjeu/data/raw/com_antitr...,html,True
3,900a0c22355f83c8f890,AT.32150,/home/edik/projects/eccjeu/data/raw/com_antitr...,html,True
4,7552bec838dfb18868d0,AT.32450,/home/edik/projects/eccjeu/data/raw/com_antitr...,pdf,True
5,a5301ee87dec66d028b8,AT.32948,/home/edik/projects/eccjeu/data/raw/com_antitr...,html,True
6,e822dc325be9e9ca1831,AT.39003,/home/edik/projects/eccjeu/data/raw/com_antitr...,html,True
7,c6ecf0efa3ce2b8bb518,AT.39004,/home/edik/projects/eccjeu/data/raw/com_antitr...,html,True
8,7d3f0e0da7f2774edf86,AT.39046,/home/edik/projects/eccjeu/data/raw/com_antitr...,html,True
9,0b95492a91ea3f6a66ef,AT.33585,/home/edik/projects/eccjeu/data/raw/com_antitr...,pdf,True


## 3. Text decoding, cleaning, diagnostics, and quality scoring

In [4]:

MOJIBAKE_PATTERNS = [
    "Ã", "Â", "â€™", "â€œ", "â€", "ðŸ", "�",
]

LEGAL_TERMS = {
    "en": ["commission", "decision", "court", "article", "applicant", "undertaking", "competition"],
    "de": ["kommission", "entscheidung", "gericht", "artikel", "kläger", "unternehmen", "wettbewerb"],
    "fr": ["commission", "décision", "cour", "article", "requérant", "entreprise", "concurrence"],
    "it": ["commissione", "decisione", "corte", "articolo", "ricorrente", "impresa", "concorrenza"],
}

COMMON_WORDS = {
    "en": ["the", "of", "and", "to", "in", "that", "for", "on", "with"],
    "de": ["der", "die", "das", "und", "von", "zu", "in", "für", "mit"],
    "fr": ["de", "la", "le", "et", "des", "les", "du", "pour", "dans"],
    "it": ["di", "la", "il", "e", "del", "della", "per", "in", "con"],
}


def read_file_bytes(path: Path) -> bytes:
    return path.read_bytes()


def normalize_unicode(text: str) -> str:
    text = unicodedata.normalize("NFKC", text or "")
    replacements = {
        "\u00a0": " ",
        "\u200b": "",
        "\ufeff": "",
        "\r\n": "\n",
        "\r": "\n",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def score_decoded_text(text: str) -> float:
    if not text:
        return -100.0
    n = len(text)
    replacement_ratio = text.count("�") / max(n, 1)
    control_ratio = sum(
        1 for ch in text
        if unicodedata.category(ch).startswith("C") and ch not in "\n\t"
    ) / max(n, 1)
    mojibake_ratio = sum(text.count(p) for p in MOJIBAKE_PATTERNS) / max(n, 1)
    printable_ratio = sum(ch.isprintable() or ch in "\n\t" for ch in text) / max(n, 1)
    return (
        printable_ratio * 30
        - replacement_ratio * 300
        - control_ratio * 200
        - mojibake_ratio * 180
    )


def extract_declared_html_encoding(data: bytes) -> Optional[str]:
    head = data[:8192]
    ascii_head = head.decode("ascii", errors="ignore")
    patterns = [
        r'<meta[^>]+charset\s*=\s*["\']?\s*([A-Za-z0-9._-]+)',
        r'<meta[^>]+content\s*=\s*["\'][^"\']*charset\s*=\s*([A-Za-z0-9._-]+)',
        r'<\?xml[^>]+encoding\s*=\s*["\']([^"\']+)',
    ]
    for pattern in patterns:
        match = re.search(pattern, ascii_head, flags=re.I)
        if match:
            return match.group(1).strip()
    return None


def decode_bytes_smart(data: bytes, is_html: bool = False) -> Tuple[str, Dict[str, Any]]:
    candidates: List[Tuple[str, str, str]] = []

    if data.startswith(b"\xef\xbb\xbf"):
        candidates.append(("utf-8-sig", "bom", data.decode("utf-8-sig", errors="replace")))
    elif data.startswith((b"\xff\xfe", b"\xfe\xff")):
        candidates.append(("utf-16", "bom", data.decode("utf-16", errors="replace")))

    declared = extract_declared_html_encoding(data) if is_html else None
    if declared:
        try:
            candidates.append((declared, "declared", data.decode(declared, errors="replace")))
        except Exception:
            pass

    if is_html and UnicodeDammit is not None:
        try:
            dammit = UnicodeDammit(data, is_html=True, smart_quotes_to=None)
            if dammit.unicode_markup:
                candidates.append((
                    dammit.original_encoding or "unknown",
                    "unicode_dammit",
                    dammit.unicode_markup,
                ))
        except Exception:
            pass

    try:
        candidates.append(("utf-8", "strict_utf8", data.decode("utf-8", errors="strict")))
    except UnicodeDecodeError:
        pass

    if charset_from_bytes is not None:
        try:
            matches = charset_from_bytes(data)
            for match in list(matches)[:4]:
                text = str(match)
                candidates.append((
                    getattr(match, "encoding", None) or "unknown",
                    "charset_normalizer",
                    text,
                ))
        except Exception:
            pass

    for encoding in ["cp1252", "latin-1"]:
        try:
            candidates.append((encoding, "fallback", data.decode(encoding, errors="replace")))
        except Exception:
            pass

    if not candidates:
        candidates.append(("utf-8", "last_resort", data.decode("utf-8", errors="replace")))

    unique = {}
    for enc, source, text in candidates:
        key = hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()
        unique.setdefault(key, (enc, source, text))

    scored = []
    for enc, source, text in unique.values():
        base_score = score_decoded_text(text)
        repaired = None
        repaired_score = None

        if ftfy_fix_text is not None:
            try:
                repaired = ftfy_fix_text(text)
                repaired_score = score_decoded_text(repaired)
            except Exception:
                repaired = None

        use_repaired = (
            repaired is not None
            and repaired != text
            and repaired_score is not None
            and repaired_score > base_score + 0.5
        )
        final_text = repaired if use_repaired else text
        final_score = repaired_score if use_repaired else base_score

        scored.append({
            "encoding": enc,
            "encoding_source": source,
            "text": final_text,
            "decode_score": float(final_score),
            "ftfy_applied": bool(use_repaired),
            "declared_encoding": declared or "",
        })

    best = max(scored, key=lambda x: x["decode_score"])
    meta = {k: v for k, v in best.items() if k != "text"}
    meta["decode_candidate_count"] = len(scored)
    return normalize_unicode(best["text"]), meta


def clean_text_readable(text: str) -> str:
    text = normalize_unicode(text)
    text = text.replace("\u00ad", "")
    text = re.sub(
        r"(?<=[A-Za-zÀ-ÖØ-öø-ÿ])[-‐‑]\s*\n\s*(?=[a-zà-öø-ÿ])",
        "",
        text,
    )
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip() + "\n" if text.strip() else ""


def clean_text_for_regex_and_llm(text: str) -> str:
    text = clean_text_readable(text)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip() + "\n" if text.strip() else ""


def repeated_character_ratio(text: str) -> float:
    if not text:
        return 1.0
    repeated = sum(len(m.group(0)) for m in re.finditer(r"(.)\1{5,}", text, flags=re.S))
    return repeated / max(len(text), 1)


def duplicate_line_ratio(text: str) -> float:
    lines = [
        re.sub(r"\s+", " ", line).strip().lower()
        for line in text.splitlines()
        if len(re.sub(r"\s+", " ", line).strip()) >= 20
    ]
    if not lines:
        return 0.0
    counts = Counter(lines)
    duplicate_instances = sum(count - 1 for count in counts.values() if count > 1)
    return duplicate_instances / max(len(lines), 1)


def private_use_ratio(text: str) -> float:
    if not text:
        return 0.0
    return sum(
        0xE000 <= ord(ch) <= 0xF8FF
        for ch in text
    ) / max(len(text), 1)


def token_diagnostics(text: str) -> Dict[str, float]:
    tokens = re.findall(r"\b[\wÀ-ÖØ-öø-ÿ]+\b", text, flags=re.UNICODE)
    if not tokens:
        return {
            "one_char_token_ratio": 1.0,
            "very_long_token_ratio": 1.0,
            "no_vowel_token_ratio": 1.0,
            "mixed_alnum_token_ratio": 0.0,
        }

    vowels = set("aeiouyäöüàâæçéèêëîïôœùûüÿ")
    one_char = sum(len(t) == 1 for t in tokens)
    very_long = sum(len(t) > 30 for t in tokens)
    no_vowel = sum(
        len(t) >= 5 and not any(ch.lower() in vowels for ch in t)
        for t in tokens
    )
    mixed = sum(any(ch.isalpha() for ch in t) and any(ch.isdigit() for ch in t) for t in tokens)

    n = len(tokens)
    return {
        "one_char_token_ratio": one_char / n,
        "very_long_token_ratio": very_long / n,
        "no_vowel_token_ratio": no_vowel / n,
        "mixed_alnum_token_ratio": mixed / n,
    }


def language_plausibility(text: str) -> Tuple[float, str]:
    words = re.findall(r"\b[\wÀ-ÖØ-öø-ÿ]+\b", text.lower(), flags=re.UNICODE)
    if not words:
        return 0.0, ""

    counts = Counter(words)
    scores = {}
    for lang in COMMON_WORDS:
        common_hits = sum(counts[w] for w in COMMON_WORDS[lang])
        legal_hits = sum(counts[w] for w in LEGAL_TERMS[lang])
        scores[lang] = min(1.0, (common_hits * 0.5 + legal_hits * 2.0) / max(len(words) * 0.01, 1))

    best_lang = max(scores, key=scores.get)
    return float(scores[best_lang]), best_lang


def text_quality_profile(raw_text: str, clean_text: str, meta: Dict[str, Any], file_format: str) -> Dict[str, Any]:
    raw_text = raw_text or ""
    clean_text = clean_text or ""
    n = len(clean_text)

    alpha_ratio = sum(ch.isalpha() for ch in clean_text) / max(n, 1)
    digit_ratio = sum(ch.isdigit() for ch in clean_text) / max(n, 1)
    printable_ratio = sum(ch.isprintable() or ch in "\n\t" for ch in clean_text) / max(n, 1)
    replacement_ratio = clean_text.count("�") / max(n, 1)
    mojibake_ratio = sum(clean_text.count(p) for p in MOJIBAKE_PATTERNS) / max(n, 1)
    control_ratio = sum(
        1 for ch in clean_text
        if unicodedata.category(ch).startswith("C") and ch not in "\n\t"
    ) / max(n, 1)

    token_stats = token_diagnostics(clean_text)
    lang_score, detected_language = language_plausibility(clean_text)
    repeat_ratio = repeated_character_ratio(clean_text)
    duplicate_ratio = duplicate_line_ratio(clean_text)
    pua_ratio = private_use_ratio(clean_text)

    length_component = min(25.0, n / 800.0)
    score = (
        length_component
        + printable_ratio * 15
        + min(alpha_ratio / 0.65, 1.0) * 10
        + lang_score * 20
        - replacement_ratio * 250
        - mojibake_ratio * 180
        - control_ratio * 250
        - pua_ratio * 250
        - repeat_ratio * 100
        - duplicate_ratio * 20
        - token_stats["one_char_token_ratio"] * 18
        - token_stats["very_long_token_ratio"] * 80
        - token_stats["no_vowel_token_ratio"] * 18
    )

    reasons = []
    if n < MIN_CHARS_ANY:
        reasons.append("very_short_text")
    elif file_format == "pdf" and n < MIN_PDF_CHARS_GOOD:
        reasons.append("short_pdf_text")
    if replacement_ratio > 0.002:
        reasons.append("replacement_characters")
    if mojibake_ratio > 0.0005:
        reasons.append("possible_mojibake")
    if pua_ratio > 0.0005:
        reasons.append("private_use_characters")
    if repeat_ratio > 0.01:
        reasons.append("repeated_character_garbage")
    if duplicate_ratio > 0.20:
        reasons.append("many_duplicate_lines")
    if token_stats["one_char_token_ratio"] > 0.30:
        reasons.append("many_one_character_tokens")
    if token_stats["very_long_token_ratio"] > 0.01:
        reasons.append("many_very_long_tokens")
    if lang_score < 0.10 and n > 1000:
        reasons.append("low_language_plausibility")

    if n < MIN_CHARS_ANY or score < MIN_SCORE_ACCEPTABLE:
        quality_flag = "failed"
    elif score < MIN_SCORE_OK or reasons:
        quality_flag = "fishy"
    else:
        quality_flag = "ok"

    return {
        "quality_score": round(float(score), 3),
        "quality_flag": quality_flag,
        "quality_reasons": "|".join(reasons),
        "n_chars_raw": len(raw_text),
        "n_chars_clean": n,
        "n_pages": meta.get("n_pages"),
        "alpha_ratio": round(alpha_ratio, 6),
        "digit_ratio": round(digit_ratio, 6),
        "printable_ratio": round(printable_ratio, 6),
        "replacement_ratio": round(replacement_ratio, 8),
        "mojibake_ratio": round(mojibake_ratio, 8),
        "private_use_ratio": round(pua_ratio, 8),
        "repeated_character_ratio": round(repeat_ratio, 8),
        "duplicate_line_ratio": round(duplicate_ratio, 8),
        "language_score": round(lang_score, 6),
        "detected_language": detected_language,
        **{k: round(v, 8) for k, v in token_stats.items()},
    }


## 4. HTML, TXT, PDF, Poppler, and MinerU candidate extraction

In [5]:

HTML_REMOVE_SELECTORS = [
    "script", "style", "noscript", "nav", "footer", "header", "form", "aside",
    "[class*='cookie']", "[id*='cookie']", "[class*='share']",
    "[class*='language']", "[aria-label*='language']",
    "[class*='pagination']", "[class*='print']",
]

HTML_CANDIDATE_SELECTORS = [
    "main", "article", "#document1", "#TexteOnly", "#text",
    ".document-content", ".document", ".content", "body",
]


def html_container_score(tag) -> float:
    text = tag.get_text(" ", strip=True)
    if not text:
        return -100.0

    links = tag.find_all("a")
    link_text_chars = sum(len(a.get_text(" ", strip=True)) for a in links)
    link_density = link_text_chars / max(len(text), 1)
    paragraphs = len(tag.find_all(["p", "div", "li", "table"]))
    lang_score, _ = language_plausibility(text)

    return (
        min(len(text), 150_000) / 1500
        + min(paragraphs, 200) * 0.08
        + lang_score * 15
        - link_density * 40
    )


def extract_text_from_html(path: Path) -> Tuple[str, Dict[str, Any]]:
    data = read_file_bytes(path)
    html, decode_meta = decode_bytes_smart(data, is_html=True)

    if BeautifulSoup is None:
        text = re.sub(r"<script.*?</script>|<style.*?</style>", " ", html, flags=re.I | re.S)
        text = re.sub(r"<[^>]+>", " ", text)
        return normalize_unicode(text), {
            "method": "html_regex",
            "n_pages": None,
            **decode_meta,
        }

    soup = BeautifulSoup(html, "html.parser")

    for selector in HTML_REMOVE_SELECTORS:
        try:
            for tag in soup.select(selector):
                tag.decompose()
        except Exception:
            pass

    candidates = []
    for selector in HTML_CANDIDATE_SELECTORS:
        try:
            for tag in soup.select(selector):
                text = tag.get_text("\n", strip=True)
                if len(text) >= 100:
                    candidates.append((html_container_score(tag), selector, text))
        except Exception:
            pass

    if candidates:
        _, selected_selector, text = max(candidates, key=lambda x: x[0])
    else:
        selected_selector = "document"
        text = soup.get_text("\n", strip=True)

    return normalize_unicode(text), {
        "method": "html_bs4_best_container",
        "n_pages": None,
        "html_selected_container": selected_selector,
        "html_candidate_count": len(candidates),
        **decode_meta,
    }


def extract_text_from_plain(path: Path) -> Tuple[str, Dict[str, Any]]:
    text, decode_meta = decode_bytes_smart(read_file_bytes(path), is_html=False)
    return text, {"method": "plain_text_smart_decode", "n_pages": None, **decode_meta}


def extract_pdf_pymupdf_text(path: Path) -> Tuple[str, Dict[str, Any]]:
    if fitz is None:
        raise RuntimeError("PyMuPDF is not installed.")

    chunks = []
    with fitz.open(path) as doc:
        for page_no, page in enumerate(doc, start=1):
            text = page.get_text("text", sort=True) or ""
            chunks.append(f"\n\n[page {page_no}]\n{text}")
        n_pages = len(doc)

    return normalize_unicode("".join(chunks)), {
        "method": "pdf_pymupdf_text_sorted",
        "n_pages": n_pages,
    }


def extract_pdf_pymupdf_blocks(path: Path) -> Tuple[str, Dict[str, Any]]:
    if fitz is None:
        raise RuntimeError("PyMuPDF is not installed.")

    chunks = []
    with fitz.open(path) as doc:
        for page_no, page in enumerate(doc, start=1):
            blocks = page.get_text("blocks", sort=True) or []
            block_texts = []
            for block in blocks:
                if len(block) >= 5:
                    text = str(block[4] or "").strip()
                    if text:
                        block_texts.append(text)
            chunks.append(f"\n\n[page {page_no}]\n" + "\n\n".join(block_texts))
        n_pages = len(doc)

    return normalize_unicode("".join(chunks)), {
        "method": "pdf_pymupdf_blocks_sorted",
        "n_pages": n_pages,
    }


def extract_pdf_pdftotext(path: Path) -> Tuple[str, Dict[str, Any]]:
    if not PDFTOTEXT_CLI:
        raise RuntimeError("pdftotext is not installed or not on PATH.")

    with tempfile.TemporaryDirectory(prefix="pdftotext_") as tmpdir:
        out_txt = Path(tmpdir) / "output.txt"
        cmd = [PDFTOTEXT_CLI, "-layout", "-enc", "UTF-8", str(path), str(out_txt)]
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        if proc.returncode != 0:
            raise RuntimeError(proc.stderr[-2000:] or "pdftotext failed")
        text = out_txt.read_text(encoding="utf-8", errors="replace")

    n_pages = None
    if fitz is not None:
        try:
            with fitz.open(path) as doc:
                n_pages = len(doc)
        except Exception:
            pass

    return normalize_unicode(text), {
        "method": "pdf_pdftotext_layout",
        "n_pages": n_pages,
    }


def format_mineru_command(input_pdf: Path, output_dir: Path) -> List[str]:
    return [
        part.format(input_pdf=str(input_pdf), output_dir=str(output_dir))
        for part in MINERU_COMMAND_TEMPLATE
    ]


def run_mineru(input_pdf: Path, output_dir: Path) -> Dict[str, Any]:
    output_dir.mkdir(parents=True, exist_ok=True)
    cmd = format_mineru_command(input_pdf, output_dir)

    try:
        env = os.environ.copy()
        env.setdefault("ORT_LOG_SEVERITY_LEVEL", "3")
        proc = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=MINERU_TIMEOUT_SECONDS,
            env=env,
        )
        return {
            "ok": proc.returncode == 0,
            "returncode": proc.returncode,
            "cmd": " ".join(cmd),
            "stdout_tail": proc.stdout[-2000:],
            "stderr_tail": proc.stderr[-2000:],
        }
    except Exception as exc:
        return {
            "ok": False,
            "returncode": None,
            "cmd": " ".join(cmd),
            "error": repr(exc),
        }


def find_mineru_text(output_dir: Path) -> Optional[Path]:
    candidates = []
    for pattern in ["**/*.md", "**/*.txt"]:
        candidates.extend(output_dir.glob(pattern))
    candidates = [p for p in candidates if p.is_file() and p.stat().st_size > 0]
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_size)


def extract_pdf_mineru(path: Path) -> Tuple[str, Dict[str, Any]]:
    with tempfile.TemporaryDirectory(prefix="mineru_") as tmpdir:
        out_dir = Path(tmpdir)
        result = run_mineru(path, out_dir)
        if not result.get("ok"):
            raise RuntimeError(result.get("error") or result.get("stderr_tail") or "MinerU failed")

        text_file = find_mineru_text(out_dir)
        if text_file is None:
            raise RuntimeError("MinerU completed but produced no non-empty markdown/text file.")

        text = text_file.read_text(encoding="utf-8", errors="replace")

    n_pages = None
    if fitz is not None:
        try:
            with fitz.open(path) as doc:
                n_pages = len(doc)
        except Exception:
            pass

    return normalize_unicode(text), {
        "method": "pdf_mineru",
        "n_pages": n_pages,
    }


def make_candidate(
    document_id: str,
    method: str,
    raw_text: str,
    meta: Dict[str, Any],
    file_format: str,
) -> Dict[str, Any]:
    readable_text = clean_text_readable(raw_text)
    regex_text = clean_text_for_regex_and_llm(raw_text)
    quality = text_quality_profile(raw_text, regex_text, meta, file_format)

    return {
        "document_id": document_id,
        "candidate_method": method,
        "raw_text": raw_text,
        "readable_text": readable_text,
        "clean_text": regex_text,
        "meta": meta,
        **quality,
    }


def generate_candidates(row: Dict[str, Any]) -> Tuple[List[Dict[str, Any]], List[str]]:
    path = Path(str(row["raw_file_path"]))
    file_format = row.get("file_format", "")
    document_id = row["document_id"]
    candidates = []
    errors = []

    def attempt(method_name, extractor):
        try:
            raw_text, meta = extractor(path)
            candidates.append(make_candidate(
                document_id=document_id,
                method=method_name,
                raw_text=raw_text,
                meta=meta,
                file_format=file_format,
            ))
        except Exception as exc:
            errors.append(f"{method_name}: {repr(exc)}")

    if file_format == "html":
        attempt("html_best_container", extract_text_from_html)
    elif file_format == "txt":
        attempt("plain_text_smart_decode", extract_text_from_plain)
    elif file_format == "pdf":
        attempt("pdf_pymupdf_text_sorted", extract_pdf_pymupdf_text)
        attempt("pdf_pymupdf_blocks_sorted", extract_pdf_pymupdf_blocks)

        if RUN_PDFTOTEXT and PDFTOTEXT_CLI:
            attempt("pdf_pdftotext_layout", extract_pdf_pdftotext)

        best_native = max(
            candidates,
            key=candidate_preference,
            default=None,
        )
        best_native_score = (
            float(
                best_native.get(
                    "quality_score",
                    -999,
                )
            )
            if best_native
            else -999
        )
        best_native_reasons = {
            reason
            for reason in str(
                best_native.get(
                    "quality_reasons",
                    "",
                )
                if best_native
                else ""
            ).split("|")
            if reason
        }

        should_run_ocr = (
            RUN_MINERU
            and MINERU_CLI is not None
            and (
                RUN_MINERU_FOR_ALL_PDFS
                or best_native is None
                or best_native.get(
                    "quality_flag"
                ) == "failed"
                or best_native_score
                < MIN_NATIVE_SCORE_TO_RUN_OCR
                or bool(
                    best_native_reasons
                    & SERIOUS_OCR_REASONS
                )
            )
        )
        if should_run_ocr:
            attempt("pdf_mineru", extract_pdf_mineru)
    else:
        errors.append(f"unsupported_file_format: {file_format}")

    return candidates, errors


## 5. Candidate selection, canonical output, and manifests

In [6]:

def candidate_preference(candidate: Dict[str, Any]) -> Tuple[float, int]:
    # Quality score dominates. This tie-breaker prefers less destructive/native methods.
    method_order = {
        "html_best_container": 5,
        "plain_text_smart_decode": 5,
        "pdf_pymupdf_text_sorted": 4,
        "pdf_pdftotext_layout": 3,
        "pdf_pymupdf_blocks_sorted": 2,
        "pdf_mineru": 1,
    }
    return (
        float(candidate.get("quality_score", -999)),
        method_order.get(candidate.get("candidate_method", ""), 0),
    )


def save_candidate_files(document_id: str, candidate: Dict[str, Any]) -> Dict[str, str]:
    method = safe_slug(candidate["candidate_method"], max_len=50)
    doc_dir = CANDIDATE_DATA_DIR / document_id
    doc_dir.mkdir(parents=True, exist_ok=True)

    raw_path = doc_dir / f"{method}__raw.txt"
    readable_path = doc_dir / f"{method}__readable.txt"
    regex_path = doc_dir / f"{method}__regex.txt"

    raw_path.write_text(candidate.get("raw_text", ""), encoding="utf-8")
    readable_path.write_text(candidate.get("readable_text", ""), encoding="utf-8")
    regex_path.write_text(candidate.get("clean_text", ""), encoding="utf-8")

    return {
        "candidate_raw_path": str(raw_path),
        "candidate_readable_path": str(readable_path),
        "candidate_regex_path": str(regex_path),
    }


def write_markdown(
    row: Dict[str, Any],
    clean_text: str,
    markdown_path: Path,
    selected_candidate: Dict[str, Any],
) -> None:
    meta = {
        "document_id": row.get("document_id"),
        "source": "com_antitrust",
        "title": row.get("title"),
        "item_date": row.get("item_date"),
        "case_number": row.get("case_number"),
        "identifier": row.get("identifier"),
        "download_key": row.get("download_key"),
        "file_format": row.get("file_format"),
        "raw_file_path": row.get("raw_file_path"),
        "download_url": row.get("download_url"),
        "extraction_version": EXTRACTION_VERSION,
        "selected_method": selected_candidate.get("candidate_method"),
        "quality_score": selected_candidate.get("quality_score"),
        "quality_flag": selected_candidate.get("quality_flag"),
        "quality_reasons": selected_candidate.get("quality_reasons"),
        "created_at": datetime.now().isoformat(timespec="seconds"),
    }
    header = (
        "---\n"
        + "\n".join(f"{key}: {json.dumps(value, ensure_ascii=False)}" for key, value in meta.items())
        + "\n---\n\n"
    )
    markdown_path.write_text(header + clean_text, encoding="utf-8")


def existing_output_is_current(document_id: str) -> bool:
    if FORCE_REEXTRACT or not CLEAN_FILE_MANIFEST_PATH.exists():
        return False

    try:
        old = pd.read_csv(CLEAN_FILE_MANIFEST_PATH, low_memory=False)
    except Exception:
        return False

    hit = old[old["document_id"].astype(str).eq(str(document_id))]
    if hit.empty:
        return False

    row = hit.iloc[-1]
    version_ok = int(row.get("extraction_version", 0) or 0) == EXTRACTION_VERSION
    clean_path = Path(str(row.get("clean_text_path", "")))
    markdown_path = Path(str(row.get("markdown_path", "")))
    return version_ok and clean_path.exists() and markdown_path.exists()


def process_one(row: Dict[str, Any]) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
    document_id = row["document_id"]
    clean_text_path = CLEAN_DATA_DIR / f"{document_id}.txt"
    readable_text_path = CLEAN_DATA_DIR / f"{document_id}__readable.txt"
    markdown_path = CLEAN_DATA_DIR / f"{document_id}.md"

    base = {
        **row,
        "processed_at": datetime.now().isoformat(timespec="seconds"),
        "extraction_version": EXTRACTION_VERSION,
        "clean_text_path": str(clean_text_path),
        "readable_text_path": str(readable_text_path),
        "markdown_path": str(markdown_path),
    }

    if not row.get("cleaning_target", False):
        result = {
            **base,
            "clean_success": False,
            "selected_method": "",
            "extraction_engine": "",
            "quality_flag": "skipped",
            "quality_score": None,
            "n_chars_clean": 0,
            "candidate_count": 0,
            "selection_reason": "not_a_cleaning_target",
            "error": "not_a_cleaning_target",
        }
        return result, []

    if existing_output_is_current(document_id):
        old = pd.read_csv(CLEAN_FILE_MANIFEST_PATH, low_memory=False)
        hit = old[old["document_id"].astype(str).eq(str(document_id))].iloc[-1].to_dict()
        hit["processed_at"] = datetime.now().isoformat(timespec="seconds")
        hit["selection_reason"] = "existing_current_v3_output"
        return hit, []

    candidates, errors = generate_candidates(row)

    candidate_rows = []
    for candidate in candidates:
        paths = {}
        if SAVE_ALL_CANDIDATES:
            paths = save_candidate_files(document_id, candidate)

        candidate_rows.append({
            "document_id": document_id,
            "raw_file_path": row.get("raw_file_path"),
            "file_format": row.get("file_format"),
            "candidate_method": candidate.get("candidate_method"),
            "quality_score": candidate.get("quality_score"),
            "quality_flag": candidate.get("quality_flag"),
            "quality_reasons": candidate.get("quality_reasons"),
            "n_chars_raw": candidate.get("n_chars_raw"),
            "n_chars_clean": candidate.get("n_chars_clean"),
            "n_pages": candidate.get("n_pages"),
            "detected_language": candidate.get("detected_language"),
            "language_score": candidate.get("language_score"),
            "replacement_ratio": candidate.get("replacement_ratio"),
            "mojibake_ratio": candidate.get("mojibake_ratio"),
            "private_use_ratio": candidate.get("private_use_ratio"),
            "repeated_character_ratio": candidate.get("repeated_character_ratio"),
            "duplicate_line_ratio": candidate.get("duplicate_line_ratio"),
            "one_char_token_ratio": candidate.get("one_char_token_ratio"),
            "very_long_token_ratio": candidate.get("very_long_token_ratio"),
            "no_vowel_token_ratio": candidate.get("no_vowel_token_ratio"),
            "meta_json": json.dumps(candidate.get("meta", {}), ensure_ascii=False),
            **paths,
        })

    if not candidates:
        result = {
            **base,
            "clean_success": False,
            "selected_method": "",
            "extraction_engine": "",
            "quality_flag": "failed",
            "quality_score": None,
            "n_chars_clean": 0,
            "candidate_count": 0,
            "selection_reason": "no_candidate_succeeded",
            "error": " | ".join(errors),
        }
        return result, candidate_rows

    selected = max(candidates, key=candidate_preference)
    clean_text = selected.get("clean_text", "")
    readable_text = selected.get("readable_text", "")

    clean_text_path.write_text(clean_text, encoding="utf-8")
    readable_text_path.write_text(readable_text, encoding="utf-8")
    write_markdown(row, clean_text, markdown_path, selected)

    score_listing = "; ".join(
        f"{c['candidate_method']}={c['quality_score']:.2f}"
        for c in sorted(candidates, key=candidate_preference, reverse=True)
    )
    selection_reason = (
        f"highest_quality_score; selected={selected['candidate_method']}; "
        f"candidate_scores={score_listing}"
    )

    clean_success = bool(
        clean_text
        and selected.get("quality_flag") != "failed"
        and selected.get("quality_score", -999) >= MIN_SCORE_ACCEPTABLE
    )

    selected_meta = selected.get("meta", {})
    selected_reasons = {
        reason
        for reason in str(
            selected.get(
                "quality_reasons",
                "",
            )
        ).split("|")
        if reason
    }
    serious_review_reasons = {
        "very_short_text",
        "replacement_characters",
        "possible_mojibake",
        "private_use_characters",
        "repeated_character_garbage",
        "low_language_plausibility",
    }
    needs_manual_review = bool(
        selected_reasons
        & serious_review_reasons
    ) or selected.get(
        "quality_flag"
    ) == "failed"

    ranked_candidates = sorted(
        candidates,
        key=candidate_preference,
        reverse=True,
    )
    second_best_score = (
        ranked_candidates[1].get(
            "quality_score"
        )
        if len(ranked_candidates) > 1
        else None
    )
    selection_margin = (
        selected.get(
            "quality_score"
        ) - second_best_score
        if second_best_score is not None
        else None
    )

    result = {
        **base,
        "clean_success": clean_success,
        "selected_method": selected.get("candidate_method"),
        # Keep legacy-compatible column name:
        "extraction_engine": selected.get("candidate_method"),
        "quality_score": selected.get("quality_score"),
        "quality_flag": selected.get("quality_flag"),
        "quality_reasons": selected.get("quality_reasons"),
        "needs_manual_review": needs_manual_review,
        "n_chars_raw": selected.get("n_chars_raw"),
        "n_chars_clean": selected.get("n_chars_clean"),
        "n_pages": selected.get("n_pages"),
        "detected_language": selected.get("detected_language"),
        "language_score": selected.get("language_score"),
        "replacement_ratio": selected.get("replacement_ratio"),
        "mojibake_ratio": selected.get("mojibake_ratio"),
        "private_use_ratio": selected.get("private_use_ratio"),
        "repeated_character_ratio": selected.get("repeated_character_ratio"),
        "duplicate_line_ratio": selected.get("duplicate_line_ratio"),
        "one_char_token_ratio": selected.get("one_char_token_ratio"),
        "very_long_token_ratio": selected.get("very_long_token_ratio"),
        "no_vowel_token_ratio": selected.get("no_vowel_token_ratio"),
        "candidate_count": len(candidates),
        "candidate_methods": "|".join(c["candidate_method"] for c in candidates),
        "candidate_scores": score_listing,
        "second_best_score": second_best_score,
        "selection_margin": selection_margin,
        "selection_confidence": (
            "single_candidate"
            if selection_margin is None
            else "high"
            if selection_margin >= 5
            else "medium"
            if selection_margin >= 1
            else "low"
        ),
        "selection_reason": selection_reason,
        "selected_encoding": selected_meta.get("encoding", ""),
        "selected_encoding_source": selected_meta.get("encoding_source", ""),
        "selected_ftfy_applied": selected_meta.get("ftfy_applied", False),
        "mineru_used": selected.get("candidate_method") == "pdf_mineru",
        "candidate_errors": " | ".join(errors),
        "error": "" if clean_success else "selected_candidate_below_quality_threshold",
    }

    return result, candidate_rows


target_rows = work_manifest.to_dict("records")
if PROCESS_LIMIT is not None:
    target_rows = target_rows[:PROCESS_LIMIT]
results = []
candidate_results = []

for row in tqdm(target_rows, desc="Cleaning COM antitrust files v3"):
    result, candidate_rows = process_one(row)
    results.append(result)
    candidate_results.extend(candidate_rows)

clean_file_manifest = pd.DataFrame(results)
candidate_manifest = pd.DataFrame(candidate_results)

essential_cols = [
    "document_id", "download_key", "identifier", "case_number",
    "item_date", "item_label", "title", "file_format",
    "download_success", "download_status", "raw_file_exists", "raw_file_path",
    "cleaning_target", "clean_success", "extraction_version",
    "quality_flag", "quality_score", "quality_reasons", "needs_manual_review",
    "n_chars_raw", "n_chars_clean", "n_pages",
    "selected_method", "extraction_engine", "mineru_used",
    "candidate_count", "candidate_methods", "candidate_scores",
    "second_best_score", "selection_margin", "selection_confidence",
    "selection_reason", "candidate_errors",
    "detected_language", "language_score",
    "replacement_ratio", "mojibake_ratio", "private_use_ratio",
    "repeated_character_ratio", "duplicate_line_ratio",
    "one_char_token_ratio", "very_long_token_ratio", "no_vowel_token_ratio",
    "selected_encoding", "selected_encoding_source", "selected_ftfy_applied",
    "clean_text_path", "readable_text_path", "markdown_path",
    "processed_at", "error",
]
essential_cols = [col for col in essential_cols if col in clean_file_manifest.columns]
clean_file_manifest = clean_file_manifest[essential_cols].copy()

clean_file_manifest.to_csv(CLEAN_FILE_MANIFEST_PATH, index=False, encoding="utf-8")
try:
    clean_file_manifest.to_excel(CLEAN_FILE_MANIFEST_XLSX_PATH, index=False)
except Exception as exc:
    print(f"Warning: could not write Excel manifest: {exc}")

if not candidate_manifest.empty:
    candidate_manifest.to_csv(
        CANDIDATE_MANIFEST_PATH,
        index=False,
        encoding="utf-8",
    )
elif not CANDIDATE_MANIFEST_PATH.exists():
    pd.DataFrame(columns=[
        "document_id",
        "candidate_method",
        "quality_score",
        "quality_flag",
    ]).to_csv(
        CANDIDATE_MANIFEST_PATH,
        index=False,
        encoding="utf-8",
    )

failed = clean_file_manifest[
    ~clean_file_manifest["clean_success"].fillna(False)
].copy()
failed.to_csv(FAILED_FILE_MANIFEST_PATH, index=False, encoding="utf-8")

print("Canonical clean-file manifest CSV:", CLEAN_FILE_MANIFEST_PATH)
print("Canonical clean-file manifest XLSX:", CLEAN_FILE_MANIFEST_XLSX_PATH)
print("Candidate manifest:", CANDIDATE_MANIFEST_PATH)
print("Best-quality files:", CLEAN_DATA_DIR)
print("Candidate audit files:", CANDIDATE_DATA_DIR)
print("Rows:", len(clean_file_manifest))
print("Clean success:", int(clean_file_manifest["clean_success"].fillna(False).sum()))
print("Failed/skipped:", len(failed))
display(clean_file_manifest.head(20))


Cleaning COM antitrust files v3:   0%|          | 0/1174 [00:00<?, ?it/s]

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

Canonical clean-file manifest CSV: /home/edik/projects/eccjeu/output/com_antitrust/com_antitrust_clean_file_manifest.csv
Canonical clean-file manifest XLSX: /home/edik/projects/eccjeu/output/com_antitrust/com_antitrust_clean_file_manifest.xlsx
Candidate manifest: /home/edik/projects/eccjeu/output/com_antitrust/com_antitrust_extraction_candidate_manifest.csv
Best-quality fil

,document_id,download_key,identifier,case_number,item_date,item_label,title,file_format,download_success,download_status,...,very_long_token_ratio,no_vowel_token_ratio,selected_encoding,selected_encoding_source,selected_ftfy_applied,clean_text_path,readable_text_path,markdown_path,processed_at,error
0,com_antitrust__COM1__AT.28841__19_04_1977__Pro...,055d7039aa0f89ee3daa,COM1,AT.28841,19/04/1977,Prohibition Decision,Prohibition Decision,pdf,True,already_exists,...,0.000000,0.000285,,,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-07-16T08:24:16,
1,com_antitrust__COM2__AT.29629__14_12_1982__Pro...,432056ba775eb3636794,COM2,AT.29629,14/12/1982,Prohibition Decision,Prohibition Decision,pdf,True,already_exists,...,0.000000,0.000452,,,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-07-16T08:24:45,
2,com_antitrust__COM3__AT.30373__12_04_1999__Old...,4e58062cea9b40420c02,COM3,AT.30373,12/04/1999,Old milestones - Exemption with condition deci...,Old milestones - Exemption with condition deci...,html,True,already_exists,...,0.000000,0.000457,utf-8,declared,True,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-07-16T08:24:45,
3,com_antitrust__COM6__AT.32150__10_05_2000__Old...,900a0c22355f83c8f890,COM6,AT.32150,10/05/2000,Old milestones - Exemption with condition deci...,Old milestones - Exemption with condition deci...,html,True,already_exists,...,0.000000,0.000436,utf-8,declared,True,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-07-16T08:24:46,
4,com_antitrust__COM7__AT.32450__30_04_2004__Pro...,7552bec838dfb18868d0,COM7,AT.32450,30/04/2004,Prohibition Decision,Prohibition Decision,pdf,True,already_exists,...,0.000000,0.000482,,,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-07-16T08:24:46,
5,com_antitrust__COM8__AT.32948__21_12_1994__Pro...,a5301ee87dec66d028b8,COM8,AT.32948,21/12/1994,Prohibition Decision,Prohibition Decision,html,True,already_exists,...,0.000000,0.000990,utf-8,declared,True,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-07-16T08:24:46,
6,com_antitrust__COM9__AT.39003__13_12_2000__Pro...,e822dc325be9e9ca1831,COM9,AT.39003,13/12/2000,Prohibition Decision,Prohibition Decision,html,True,already_exists,...,0.000000,0.005440,utf-8,declared,True,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-07-16T08:24:47,
7,com_antitrust__COM10__AT.39004__13_12_2000__Pr...,c6ecf0efa3ce2b8bb518,COM10,AT.39004,13/12/2000,Prohibition Decision,Prohibition Decision,html,True,already_exists,...,0.000000,0.004931,utf-8,declared,True,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-07-16T08:24:47,
8,com_antitrust__COM11__AT.39046__13_12_2000__Pr...,7d3f0e0da7f2774edf86,COM11,AT.39046,13/12/2000,Prohibition Decision,Prohibition Decision,html,True,already_exists,...,0.000000,0.003956,utf-8,declared,True,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-07-16T08:24:48,
9,com_antitrust__COM12__AT.33585__25_11_1992__Pr...,0b95492a91ea3f6a66ef,COM12,AT.33585,25/11/1992,Prohibition Decision,Prohibition Decision,pdf,True,already_exists,...,0.000000,0.003577,,,False,/home/edik/projects/ecc

## 6. Quality summary and candidate comparison

In [7]:

if CLEAN_FILE_MANIFEST_PATH.exists():
    manifest = pd.read_csv(CLEAN_FILE_MANIFEST_PATH, low_memory=False)

    summary = (
        manifest.groupby(
            ["file_format", "quality_flag", "selected_method"],
            dropna=False,
        )
        .size()
        .reset_index(name="n")
        .sort_values(["file_format", "quality_flag", "selected_method"])
    )
    display(summary)

    review_cols = [
        "document_id", "title", "file_format",
        "selected_method", "quality_score", "quality_flag",
        "quality_reasons", "candidate_scores", "clean_text_path",
    ]
    review_cols = [c for c in review_cols if c in manifest.columns]

    display(
        manifest.sort_values(
            ["clean_success", "quality_score"],
            ascending=[True, True],
            na_position="first",
        )[review_cols].head(100)
    )
else:
    print("Run the processing cell first.")

if (
    CANDIDATE_MANIFEST_PATH.exists()
    and CANDIDATE_MANIFEST_PATH.stat().st_size > 0
):
    try:
        candidates = pd.read_csv(
            CANDIDATE_MANIFEST_PATH,
            low_memory=False,
        )
    except pd.errors.EmptyDataError:
        candidates = pd.DataFrame()
    if not candidates.empty:
        comparison = (
            candidates.groupby(
                ["file_format", "candidate_method", "quality_flag"],
                dropna=False,
            )
            .agg(
                n=("document_id", "size"),
                mean_score=("quality_score", "mean"),
                median_score=("quality_score", "median"),
                mean_chars=("n_chars_clean", "mean"),
            )
            .reset_index()
            .sort_values(["file_format", "mean_score"], ascending=[True, False])
        )
        display(comparison)


,file_format,quality_flag,selected_method,n
0,html,fishy,html_best_container,325
1,html,ok,html_best_container,255
2,pdf,fishy,pdf_mineru,33
3,pdf,fishy,pdf_pdftotext_layout,9
4,pdf,fishy,pdf_pymupdf_blocks_sorted,14
5,pdf,fishy,pdf_pymupdf_text_sorted,7
6,pdf,ok,pdf_mineru,343
7,pdf,ok,pdf_pdftotext_layout,10
8,pdf,ok,pdf_pymupdf_blocks_sorted,52
9,pdf,ok,pdf_pymupdf_text_sorted,126


,document_id,title,file_format,selected_method,quality_score,quality_flag,quality_reasons,candidate_scores,clean_text_path
46,com_antitrust__COM55__AT.35774__05_03_1997__Ex...,Exemption without condition (Reg 17/62),pdf,pdf_mineru,41.778,fishy,short_pdf_text|many_very_long_tokens,pdf_mineru=41.78; pdf_pymupdf_text_sorted=12.8...,/home/edik/projects/eccjeu/data/processed/com_...
42,com_antitrust__COM51__AT.35763__07_10_1997__Ex...,Exemption without condition (Reg 17/62),pdf,pdf_mineru,42.289,fishy,short_pdf_text|many_very_long_tokens,pdf_mineru=42.29; pdf_pymupdf_text_sorted=12.8...,/home/edik/projects/eccjeu/data/processed/com_...
71,com_antitrust__COM80__AT.36319__07_10_1997__Ex...,Exemption without condition (Reg 17/62),pdf,pdf_mineru,42.361,fishy,short_pdf_text|many_very_long_tokens,pdf_mineru=42.36; pdf_pymupdf_text_sorted=12.8...,/home/edik/projects/eccjeu/data/processed/com_...
445,com_antitrust__COM410__AT.38700__17_04_2018__S...,State Measure Decision,pdf,pdf_mineru,42.510,fishy,low_language_plausibility,pdf_mineru=42.51; pdf_pymupdf_blocks_sorted=37...,/home/edik/projects/eccjeu/data/processed/com_...
422,com_antitrust__COM403__AT.38698__16_07_2008__P...,Prohibition Decision,pdf,pdf_mineru,42.735,fishy,low_language_plausibility,pdf_mineru=42.73; pdf_pymupdf_blocks_sorted=39...,/home/edik/projects/eccjeu/data/processed/com_...
...,...,...,...,...,...,...,...,...,...
397,com_antitrust__COM395__AT.38645__31_05_2006__P...,Prohibition Decision,html,html_best_container,47.560,fishy,NaN,html_best_container=47.56,/home/edik/projects/eccjeu/data/processed/com_...
368,com_antitrust__COM382__AT.38620__03_05_2006__P...,Prohibition Decision,html,html_best_container,47.580,fishy,NaN,html_best_container=47.58,/home/edik/projects/eccjeu/data/processed/com_...
120,com_antitrust__COM203__AT.37396__14_11_2002__E...,Exemption without condition (Reg 17/62),pdf,pdf_pymupdf_blocks_sorted,47.618,fishy,NaN,pdf_pymupdf_blocks_sorted=47.62; pdf_pymupdf_t...,/home/edik/projects/eccjeu/data/processed/com_...
495,com_antitrust__COM437__AT.39129__07_10_2009__P...,Prohibition Decision,html,html_best_container,47.626,fishy,NaN,html_best_container=47.63,/home/edik/projects/eccjeu/data/processed/com_...


,file_format,candidate_method,quality_flag,n,mean_score,median_score,mean_chars
1,html,html_best_container,ok,255,63.449039,64.1660,45255.321569
0,html,html_best_container,fishy,325,49.517517,49.1770,4589.873846
12,pdf,pdf_pymupdf_text_sorted,ok,186,68.103387,68.5155,263861.596774
9,pdf,pdf_pymupdf_blocks_sorted,ok,187,68.049203,68.3160,263379.358289
6,pdf,pdf_pdftotext_layout,ok,184,67.925853,68.3020,264952.358696
3,pdf,pdf_mineru,ok,345,67.543632,68.3810,246020.794203
11,pdf,pdf_pymupdf_text_sorted,fishy,329,63.287562,65.6200,296494.635258
8,pdf,pdf_pymupdf_blocks_sorted,fishy,328,63.279412,65.5745,297603.685976
5,pdf,pdf_pdftotext_layout,fishy,331,63.091341,65.3940,294267.589124
2,pdf,pdf_mineru,fishy,37,46.028216,45.0660,50141.351351


## 7. Inspect one canonical file and all of its candidates

In [15]:

DOCUMENT_ID_TO_VIEW = "com_antitrust__COM517__AT.39396__22_07_2009__Prohibition_Decision__74f66264f9a23f9ddcd4__pdf__f1d46e37c0"
# Example:
# DOCUMENT_ID_TO_VIEW = "com_antitrust__31982D0465__html__abc123..."

if DOCUMENT_ID_TO_VIEW:
    manifest = pd.read_csv(CLEAN_FILE_MANIFEST_PATH, low_memory=False)
    hit = manifest[manifest["document_id"].astype(str).eq(str(DOCUMENT_ID_TO_VIEW))]

    if hit.empty:
        print("Document ID not found.")
    else:
        record = hit.iloc[-1].to_dict()
        keys = [
            "document_id", "title", "celex", "file_format",
            "selected_method", "quality_score", "quality_flag",
            "quality_reasons", "candidate_scores", "clean_text_path",
            "readable_text_path",
        ]
        print(json.dumps(
            {key: record.get(key) for key in keys},
            indent=2,
            ensure_ascii=False,
        ))

        canonical_path = Path(record["clean_text_path"])
        print("\n--- CANONICAL REGEX/LLM VERSION ---\n")
        print(canonical_path.read_text(encoding="utf-8", errors="replace")[:7000])

        readable_path = Path(record["readable_text_path"])
        if readable_path.exists():
            print("\n--- READABLE VERSION ---\n")
            print(readable_path.read_text(encoding="utf-8", errors="replace")[:7000])

        if CANDIDATE_MANIFEST_PATH.exists() and CANDIDATE_MANIFEST_PATH.stat().st_size > 0:
            candidate_manifest = pd.read_csv(CANDIDATE_MANIFEST_PATH, low_memory=False)
            candidate_hit = candidate_manifest[
                candidate_manifest["document_id"].astype(str).eq(str(DOCUMENT_ID_TO_VIEW))
            ].sort_values("quality_score", ascending=False)
            display(candidate_hit)
else:
    print("Set DOCUMENT_ID_TO_VIEW to inspect a cleaned file.")


{
  "document_id": "com_antitrust__COM517__AT.39396__22_07_2009__Prohibition_Decision__74f66264f9a23f9ddcd4__pdf__f1d46e37c0",
  "title": "Prohibition Decision",
  "celex": null,
  "file_format": "pdf",
  "selected_method": "pdf_mineru",
  "quality_score": 49.404,
  "quality_flag": "fishy",
  "quality_reasons": "low_language_plausibility",
  "candidate_scores": "pdf_mineru=49.40; pdf_pymupdf_text_sorted=47.49; pdf_pymupdf_blocks_sorted=47.44; pdf_pdftotext_layout=47.38",
  "clean_text_path": "/home/edik/projects/eccjeu/data/processed/com_antitrust/com_antitrust__COM517__AT.39396__22_07_2009__Prohibition_Decision__74f66264f9a23f9ddcd4__pdf__f1d46e37c0.txt",
  "readable_text_path": "/home/edik/projects/eccjeu/data/processed/com_antitrust/com_antitrust__COM517__AT.39396__22_07_2009__Prohibition_Decision__74f66264f9a23f9ddcd4__pdf__f1d46e37c0__readable.txt"
}

--- CANONICAL REGEX/LLM VERSION ---

![](images/b186f6a034521063efadfd6d921e08f53c4b3e0bf5c9ce59ea9e7b921e4d85d3.jpg)

KOMISIA EURÓ

,document_id,raw_file_path,file_format,candidate_method,quality_score,quality_flag,quality_reasons,n_chars_raw,n_chars_clean,n_pages,...,language_score,replacement_ratio,mojibake_ratio,private_use_ratio,repeated_character_ratio,duplicate_line_ratio,one_char_token_ratio,very_long_token_ratio,no_vowel_token_ratio,meta_json
1435,com_antitrust__COM517__AT.39396__22_07_2009__P...,/home/edik/projects/eccjeu/data/raw/com_antitr...,pdf,pdf_mineru,49.404,fishy,low_language_plausibility,196126,195645,96.0,...,0.090656,0.0,0.0,0.0,0.000000,0.001582,0.126927,0.000035,0.004986,"{""method"": ""pdf_mineru"", ""n_pages"": 96}"
1432,com_antitrust__COM517__AT.39396__22_07_2009__P...,/home/edik/projects/eccjeu/data/raw/com_antitr...,pdf,pdf_pymupdf_text_sorted,47.489,fishy,repeated_character_garbage,293400,244486,96.0,...,0.158575,0.0,0.0,0.0,0.032783,0.000000,0.129080,0.000000,0.004469,"{""method"": ""pdf_pymupdf_text_sorted"", ""n_pages..."
1433,com_antitrust__COM517__AT.39396__22_07_2009__P...,/home/edik/projects/eccjeu/data/raw/com_antitr...,pdf,pdf_pymupdf_blocks_sorted,47.443,fishy,repeated_character_garbage,250503,245033,96.0,...,0.158575,0.0,0.0,0.0,0.032710,0.002671,0.129080,0.000000,0.004469,"{""method"": ""pdf_pymupdf_blocks_sorted"", ""n_pag..."
1434,com_antitrust__COM517__AT.39396__22_07_2009__P...,/home/edik/projects/eccjeu/data/raw/com_antitr...,pdf,pdf_pdftotext_layout,47.384,fishy,repeated_character_garbage,286060,243685,96.0,...,0.159457,0.0,0.0,0.0,0.032891,0.000000,0.129537,0.000000,0.004494,"{""method"": ""pdf_pdftotext_layout"", ""n_pages"": 96}"


In [13]:
from pathlib import Path
import unicodedata

path = Path(
    "/home/edik/projects/eccjeu/data/processed/com_antitrust/com_antitrust__COM572__AT.39563__24_06_2015__Prohibition_Decision__41403f165bcfdb0c7c87__pdf__6ff8ddd198.txt"
)

text = path.read_text(encoding="utf-8")

for i, ch in enumerate(text):
    code = ord(ch)

    if (
        0xE000 <= code <= 0xF8FF
        or 0xF0000 <= code <= 0xFFFFD
        or 0x100000 <= code <= 0x10FFFD
    ):
        start = max(0, i - 80)
        end = min(len(text), i + 81)

        print("Code point:", f"U+{code:04X}")
        print("Name:", unicodedata.name(ch, "PRIVATE USE"))
        print("Position:", i)
        print("Context:", repr(text[start:end]))
        print("-" * 80)

Code point: U+F050
Name: PRIVATE USE
Position: 5639
Context: '2007\n\nNWE\n\n13 June 2002 – 29 Oct. 2007\n\nCEE\n\n5 Nov. 2004 – 24 Sept. 2007 Linpac \uf050 \uf050 \uf050 \uf050 \uf050\n\nVitembal \uf050 \uf050 \uf050 \uf050\n\nHuhtamäki\n\n\uf050 \uf050 \uf050\n\nSirap-Gema \uf050\n\n\uf050 \uf050\n\nCoopbox \uf050 \uf050\n\n\uf050\n'
--------------------------------------------------------------------------------
Code point: U+F050
Name: PRIVATE USE
Position: 5641
Context: '07\n\nNWE\n\n13 June 2002 – 29 Oct. 2007\n\nCEE\n\n5 Nov. 2004 – 24 Sept. 2007 Linpac \uf050 \uf050 \uf050 \uf050 \uf050\n\nVitembal \uf050 \uf050 \uf050 \uf050\n\nHuhtamäki\n\n\uf050 \uf050 \uf050\n\nSirap-Gema \uf050\n\n\uf050 \uf050\n\nCoopbox \uf050 \uf050\n\n\uf050\n\nN'
--------------------------------------------------------------------------------
Code point: U+F050
Name: PRIVATE USE
Position: 5643
Context: '\n\nNWE\n\n13 June 2002 – 29 Oct. 2007\n\nCEE\n\n5 Nov. 2004 – 24 Sept. 2007 Linpac \uf050 \uf050 \